# Phase 2 — SFT LoRA puis DPO de Qwen3-1.7B-Base

Fine-tuning supervisé (SFT) de `Qwen/Qwen3-1.7B-Base` sur le dataset anonymisé de la phase 1, avec LoRA, puis alignement par préférences (DPO) du modèle SFT.

- **Prérequis** : GPU, fichiers `data/anonymized/sft_*_{train,validation,test}.anonymized.jsonl` produits par `data_prep_MediQAl` (français, réponse unique), `data_prep_SFT_MediQAl_mcqm` (français, réponses multiples) et `data_prep_SFT_UltraMedical` (anglais). Les QCM français contiennent les propositions A à E, puis une consigne sur le nombre de bonnes réponses (une seule ou plusieurs) ; la réponse attendue donne la ou les lettres et leur texte.
- **Prérequis DPO** : fichiers `data/anonymized/dpo_mediqal_mcqm_*` et `dpo_ultramedical_*` produits par `data_prep_DPO_MediQAl_mcqm` et `data_prep_DPO_UltraMedical`.
- **Étapes** : données → modèle → LoRA → entraînement SFT → validation intermédiaire → DPO → validation DPO → modèle final fusionné.
- **Suivi** : MLflow sur DagsHub (`MLFLOW_BACKEND = "dagshub"`), consultable de partout et quelle que soit la machine d'entraînement. En local (`"local"`) : `mlflow ui --backend-store-uri sqlite:///mlflow.db` depuis la racine du projet.

In [ ]:
# Installation (Colab ou machine GPU) : décommenter si besoin
# %pip install -q torch transformers peft trl datasets accelerate mlflow dagshub pandas

## 1. Configuration

In [ ]:
import functools
import gc
import json
import math
import re
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import torch
from datasets import DatasetDict, concatenate_datasets, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

from projet_14.mediqal import LETTERS, answer_letters

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
# Données SFT de la phase 1, anonymisées : fichier -> langue
# Le SFT couvre les formats et les langues du DPO (section 7) : réponses uniques et listes en français, anglais
SFT_DATASETS = {
    "sft_mediqal_mcqu": "fr",   # QCM à réponse unique, avec cas clinique
    "sft_mediqal_mcqm": "fr",   # QCM à réponses multiples (liste), avec et sans cas clinique
    "sft_ultramedical": "en",   # questions et réponses rédigées en anglais
}

# Suivi MLflow : "dagshub" (serveur distant) ou "local" (fichier ../mlflow.db)
MLFLOW_BACKEND = "dagshub"
DAGSHUB_REPO_OWNER = "obiyohan"
DAGSHUB_REPO_NAME = "my-ml-mlflow"

# Petit run pour valider la pipeline avant l'entraînement complet
SMALL_RUN = False  # True pour un run rapide sur 100 exemples, False pour le run complet

SEED = 42
# Version des données de la phase 1 : dans le nom des runs, pour ne pas écraser les checkpoints d'une version à l'autre
# v1 : propositions absentes du prompt ; v2 : propositions dans le prompt, répartition par cas clinique,
#      DPO anglais équilibré sur la longueur ; v3 : consigne « une seule / plusieurs propositions exactes »
DATA_VERSION = "v3"
RUN_NAME = f"sft-lora-r16-{DATA_VERSION}-seed{SEED}" + ("-small" if SMALL_RUN else "")
OUTPUT_DIR = Path("../outputs") / RUN_NAME  # checkpoints et adaptateur final

set_seed(SEED)
if torch.cuda.is_available():
    device, bf16 = "cuda", torch.cuda.is_bf16_supported()  # bf16 absent sur T4 : fp16 à la place
elif torch.xpu.is_available():
    device, bf16 = "xpu", torch.xpu.is_bf16_supported()  # GPU Intel Arc
else:
    device, bf16 = "cpu", False
dtype = torch.bfloat16 if bf16 else torch.float16 if device != "cpu" else torch.float32
print(f"device={device}, dtype={dtype}")

if device == "xpu":
    # GPU Intel sous Windows : deux contournements
    # 1. Pas de compilateur C pour Triton : TRL calcule la loss en PyTorch pur
    import trl.kernels.chunked_logprob as chunked_logprob
    chunked_logprob._TRITON_AVAILABLE = False
    # 2. La norme des gradients en mode « foreach » fait planter le GPU : calcul tenseur par tenseur
    torch.nn.utils.clip_grad_norm_ = functools.partial(torch.nn.utils.clip_grad.clip_grad_norm_, foreach=False)

## 2. Données : train / validation / test

In [ ]:
# Format prompt / completion : la loss ne porte que sur la réponse de l'assistant
def to_prompt_completion(record):
    return {"prompt": record["messages"][:-1], "completion": record["messages"][-1:]}


# Chaque source garde son découpage train / validation / test ; les sources sont ensuite mélangées
parts = {split: [] for split in ("train", "validation", "test")}
for stem, lang in SFT_DATASETS.items():
    files = {split: f"../data/anonymized/{stem}_{split}.anonymized.jsonl" for split in parts}
    ds = load_dataset("json", data_files=files).select_columns(["messages"])
    for split, part in ds.items():
        parts[split].append(part.add_column("source", [stem] * len(part)).add_column("lang", [lang] * len(part)))

splits = DatasetDict({split: concatenate_datasets(p).shuffle(seed=SEED) for split, p in parts.items()})
splits = splits.map(to_prompt_completion, remove_columns=["messages"])

# Petit run : sous-échantillon (toutes sources mélangées)
if SMALL_RUN:
    for name, size in {"train": 50, "validation": 15, "test": 24}.items():
        splits[name] = splits[name].select(range(size))

pd.DataFrame({name: pd.Series(ds["source"]).value_counts() for name, ds in splits.items()}).fillna(0).astype(int)

## 3. Tokenizer et modèle

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gabarit ChatML, avec une réponse de l'assistant terminée par <|endoftext|> (eos_token) au lieu de <|im_end|>.
# Dans Qwen3-Base, <|im_start|> et <|im_end|> n'ont jamais été entraînés (embeddings à leur valeur d'initialisation)
# et LoRA ne modifie pas les embeddings : le modèle ne peut pas apprendre à produire <|im_end|>, donc à s'arrêter.
# <|endoftext|> a été appris au pré-entraînement : le modèle sait le prédire, et vLLM s'arrête dessus par défaut.
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] }}"
    "{% if message['role'] == 'assistant' %}{{ eos_token }}{% else %}{{ '<|im_end|>\n' }}{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

example = splits["train"][0]
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{model.num_parameters() / 1e9:.2f} B paramètres")

## 4. LoRA

In [ ]:
# Seules de petites matrices de rang r sont entraînées : le modèle de base reste figé
lora_config = LoraConfig(
    r=16,                         # rang des matrices d'adaptation
    lora_alpha=32,                # facteur d'échelle (souvent 2 × r)
    lora_dropout=0.05,
    target_modules="all-linear",  # couches d'attention et MLP
    task_type="CAUSAL_LM",
)

## 5. Entraînement SFT

In [ ]:
# Suivi MLflow : un run par entraînement, hyperparamètres et métriques enregistrés
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    # Jeton lu dans la variable d'environnement DAGSHUB_USER_TOKEN,
    # sinon connexion via le navigateur au premier lancement (jeton mémorisé ensuite)
    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print(f"Suivi MLflow : {mlflow.get_tracking_uri()}")

mlflow.set_experiment("qwen3-1.7b-sft-lora")
mlflow.start_run(run_name=RUN_NAME)
mlflow.log_params({"model": MODEL_NAME, "data_version": DATA_VERSION, "small_run": SMALL_RUN, "device": device, "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "lora_dropout": lora_config.lora_dropout})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    seed=SEED,
    # Optimisation
    # Une époque : au run v2 (2 époques), eval_loss était au plus bas en fin d'époque 1 puis remontait légèrement
    num_train_epochs=1,
    max_steps=30 if SMALL_RUN else -1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # batch effectif = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3 % des étapes en montée progressive du learning rate
    max_length=1024,
    gradient_checkpointing=True,  # moins de mémoire GPU, un peu plus lent
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=10 if SMALL_RUN else 100,
    save_steps=10 if SMALL_RUN else 100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=5 if SMALL_RUN else 20,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

In [ ]:
# Référence avant entraînement (adaptateur LoRA initialisé à zéro = modèle de base)
val_before = trainer.evaluate(metric_key_prefix="val_before")
val_before

In [ ]:
# Reprise après interruption : trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
train_result.metrics

In [ ]:
# Sauvegarde de l'adaptateur LoRA (meilleur checkpoint) et de la configuration du run
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "small_run": SMALL_RUN,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "dropout": lora_config.lora_dropout},
    "data": {name: len(ds) for name, ds in splits.items()},
    "training_args": training_args.to_dict(),
}
(FINAL_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")

## 6. Validation intermédiaire

1. **Loss, perplexité et token accuracy** sur validation (avant / après SFT) et test.
2. **Qualité des réponses générées** sur des exemples de test, modèle de base vs modèle SFT, sur les **lettres** données dans la réponse. Génération par lots pour bien charger le GPU.
   - `accuracy` : exactement les lettres attendues ;
   - `option_accuracy` : exactitude par proposition, part des cinq propositions correctement cochées ou laissées. Utile en `mcqm` ; en `mcqu`, une réponse vide obtient déjà 0,8 : y lire l'accuracy ;
   - `f1` : crédit partiel pour les réponses multiples. Il favorise les réponses qui cochent trop : en `mcqm`, tout cocher donne un F1 d'environ 0,7 ;
   - `extra_letters` : lettres données en trop (positif) ou en moins (négatif) par rapport au corrigé, en moyenne.

   Deux **réponses naïves**, identiques pour toutes les questions, servent de référence : `frequente` (la réponse la plus fréquente du train) et `tout_cocher` (A à E). Un modèle utile doit les battre, sur l'accuracy et l'exactitude par proposition.

La **latence** n'est pas mesurée ici : avec l'adaptateur LoRA non fusionné et Transformers, elle ne serait pas représentative. Elle sera mesurée en phase 3, sur le modèle final fusionné servi par vLLM.

In [ ]:
val_after = trainer.evaluate(metric_key_prefix="val_after")
test_metrics = trainer.evaluate(splits["test"], metric_key_prefix="test")

# token_accuracy : part des tokens de la réponse attendue que le modèle prédit correctement
# (TRL la nomme « eval_mean_token_accuracy » quel que soit le préfixe : on la renomme ici)
evaluations = {"val_before": val_before, "val_after": val_after, "test": test_metrics}
summary = pd.DataFrame({
    name: {"loss": m[f"{name}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
    for name, m in evaluations.items()
}).T
summary["perplexity"] = summary["loss"].map(math.exp)

# Même mesure sur le test, source par source : chaque format doit être appris
per_source = {}
for source in SFT_DATASETS:
    subset = splits["test"].filter(lambda example: example["source"] == source)
    if len(subset):
        m = trainer.evaluate(subset, metric_key_prefix=f"test_{source}")
        per_source[f"test_{source}"] = {"loss": m[f"test_{source}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
per_source_summary = pd.DataFrame(per_source).T
per_source_summary["perplexity"] = per_source_summary["loss"].map(math.exp)
pd.concat([summary, per_source_summary])

In [ ]:
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
STOP_IDS = [IM_END_ID, tokenizer.eos_token_id]
# Réponses mcqu : « C. texte de la proposition ». La lettre vient en premier : une réponse coupée reste évaluable.
# Le modèle de base, qui ne s'arrête pas toujours, est ainsi coupé plus tôt.
MAX_NEW_TOKENS = 64


def render(prompt):
    return tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)


@torch.no_grad()
def generate_batch(prompts, batch_size=16, max_new_tokens=MAX_NEW_TOKENS, desc=None, model=None):
    # Génération par lots (GPU bien chargé) : renvoie les réponses, dans l'ordre des prompts
    model = model or trainer.model  # par défaut, le modèle du SFT
    texts = [render(prompt) for prompt in prompts]
    # Prompts triés par longueur : moins de padding dans chaque lot, les plus longs d'abord
    # (un manque de mémoire GPU apparaît dès le premier lot)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    answers = [None] * len(texts)
    padding_side, tokenizer.padding_side = tokenizer.padding_side, "left"  # padding à gauche : la génération suit chaque prompt
    try:
        for start in tqdm(range(0, len(order), batch_size), desc=desc, unit="lot"):
            batch = order[start:start + batch_size]
            inputs = tokenizer([texts[i] for i in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
            output = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id,
            )
            decoded = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for i, answer in zip(batch, decoded):
                answers[i] = answer.strip()
    finally:
        tokenizer.padding_side = padding_side
    return answers


QUALITY_METRICS = ["accuracy", "option_accuracy", "f1", "extra_letters"]
NAIVE_ANSWERS = ["frequente", "tout_cocher"]


def letter_scores(predicted: frozenset, expected: frozenset) -> dict:
    # Mesures sur les lettres d'une réponse (ensembles de lettres A à E)
    common = len(predicted & expected)
    return {
        "accuracy": float(predicted == expected),  # exactement les lettres attendues
        # Exactitude par proposition : part des 5 propositions correctement cochées ou laissées
        "option_accuracy": sum((letter in predicted) == (letter in expected) for letter in LETTERS) / len(LETTERS),
        # F1 : crédit partiel ; favorise les réponses qui cochent trop (tout cocher : ~0,7 en mcqm)
        "f1": 2 * common / (len(predicted) + len(expected)) if common else 0.0,
        "extra_letters": len(predicted) - len(expected),  # > 0 : coche trop ; < 0 : pas assez
    }


def letters_text(letters) -> str:
    # Réponse réduite à ses lettres (« A.\nC. »), lisible par answer_letters
    return "\n".join(f"{letter}." for letter in sorted(letters))


def naive_answers(train_answers: list[str]) -> dict:
    # Réponses naïves, identiques pour toutes les questions : un modèle utile doit les battre
    most_common = Counter(answer_letters(answer) for answer in train_answers).most_common(1)[0][0]
    return {"frequente": letters_text(most_common), "tout_cocher": letters_text(LETTERS)}


def add_scores(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Ajoute <mesure>_<colonne> pour chaque colonne de réponses, comparée à df["reference"]
    expected = [answer_letters(reference) for reference in df["reference"]]
    for column in columns:
        scores = pd.DataFrame(
            [letter_scores(answer_letters(answer), e) for answer, e in zip(df[column], expected)], index=df.index
        )
        for metric in QUALITY_METRICS:
            df[f"{metric}_{column}"] = scores[metric]
    return df


def score_table(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Moyennes : une ligne par mesure, une colonne par modèle ou réponse naïve
    return pd.DataFrame({column: {metric: df[f"{metric}_{column}"].mean() for metric in QUALITY_METRICS} for column in columns})

In [ ]:
N_EVAL = 20 if SMALL_RUN else 200    # exemples de test par source
BATCH_SIZE = 16                      # à baisser en cas de manque de mémoire GPU (OutOfMemoryError)

# Sources évaluées par génération, avec leur nombre maximal de tokens générés.
# Les réponses anglaises (plusieurs centaines de tokens, rédigées) ne se comparent pas mot à mot à une référence :
# elles sont évaluées par la loss et la perplexité (cellule précédente).
GENERATION_SOURCES = {"sft_mediqal_mcqu": MAX_NEW_TOKENS, "sft_mediqal_mcqm": 192}
ANSWER_COLUMNS = ["base", "sft", *NAIVE_ANSWERS]  # modèles, puis réponses naïves de référence

trainer.model.eval()
trainer.model.config.use_cache = True  # accélère la génération

# Génération par lots, modèle de base puis modèle SFT, source par source
source_generations = []
for source, max_new_tokens in GENERATION_SOURCES.items():
    subset = splits["test"].filter(lambda example: example["source"] == source)
    subset = subset.select(range(min(N_EVAL, len(subset))))
    prompts = list(subset["prompt"])
    with trainer.model.disable_adapter():  # modèle de base, sans LoRA
        base_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"Base {source}")
    sft_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"SFT {source}")
    train_answers = [c[0]["content"] for c in splits["train"].filter(lambda example: example["source"] == source)["completion"]]
    source_generations.append(pd.DataFrame({
        "source": source,
        "question": [prompt[-1]["content"][-200:] for prompt in prompts],
        "reference": [completion[0]["content"] for completion in subset["completion"]],
        "base": base_answers,
        "sft": sft_answers,
        **naive_answers(train_answers),  # mêmes réponses pour toutes les questions de la source
    }))
generations = add_scores(pd.concat(source_generations, ignore_index=True), ANSWER_COLUMNS)
generations.to_csv(FINAL_DIR / "test_generations.csv", index=False)

# Moyennes sur toutes les sources, puis source par source
comparison = score_table(generations, ANSWER_COLUMNS)
quality_by_source = pd.concat({source: score_table(group, ANSWER_COLUMNS) for source, group in generations.groupby("source")})
display(quality_by_source)
comparison

In [ ]:
# Quelques exemples à relire
pd.set_option("display.max_colwidth", 200)
generations.groupby("source")[["source", "question", "reference", "base", "sft"]].head(3)

In [ ]:
# Métriques finales dans MLflow (les loss sont déjà envoyées par le Trainer), puis clôture du run
mlflow.log_metrics({f"{name}_token_accuracy": row["token_accuracy"] for name, row in summary.iterrows()})
mlflow.log_metrics({f"{name}_perplexity": row["perplexity"] for name, row in summary.iterrows()})
mlflow.log_metrics({
    f"test_{metric}_{model}": comparison.loc[metric, model]
    for metric in QUALITY_METRICS
    for model in comparison.columns
})
# Par source : loss, token accuracy et perplexité du test, mesures des réponses générées et des réponses naïves
mlflow.log_metrics({
    f"{name}_{metric}": value for name, row in per_source_summary.iterrows() for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_{source}_{metric}_{column}": value
    for (source, metric), row in quality_by_source.iterrows()
    for column, value in row.items()
})
mlflow.end_run()

## Lecture des résultats

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | courbes de loss d'entraînement et de validation | baissent ensemble ; si `eval_loss` remonte : surapprentissage |
| `val_before_loss`, `val_after_loss`, `test_loss` | loss avant / après SFT et sur le test | après < avant ; test proche de val |
| `*_token_accuracy` | part des tokens de la réponse correctement prédits | augmente après SFT |
| `*_perplexity` | exp(loss) | baisse après SFT |
| `test_sft_<source>_loss`, `_token_accuracy`, `_perplexity` | les mêmes mesures sur le test, source par source | chaque source s'améliore, y compris l'anglais |
| `test_accuracy_*` | réponse générée avec exactement les lettres attendues (sources françaises) | sft > base, sft > `frequente` et `tout_cocher` |
| `test_option_accuracy_*` | exactitude par proposition : part des 5 propositions correctement cochées ou laissées | sft > `frequente` et `tout_cocher` en `mcqm` ; en `mcqu`, une réponse vide obtient 0,8 : lire l'accuracy |
| `test_f1_*` | F1 sur les lettres (crédit partiel pour les réponses multiples) | sft > base ; à lire avec prudence, `tout_cocher` obtient ~0,7 en `mcqm` |
| `test_extra_letters_*` | lettres en trop (> 0) ou en moins (< 0) par rapport au corrigé | proche de 0 |
| `test_sft_<source>_<mesure>_*` | les mêmes mesures, source par source | sft bat les réponses naïves pour `mcqu` et `mcqm` |

- La courbe `eval_mean_token_accuracy` mélange toutes les évaluations (TRL ignore le préfixe) : utiliser `*_token_accuracy`.
- Les mesures portent sur les lettres : une réponse sans lettre (format non appris) compte 0.
- `*_frequente`, `*_tout_cocher` : réponses naïves, identiques pour toutes les questions (réponse la plus fréquente du train, A à E). Elles ne dépendent pas du modèle et donnent le niveau à dépasser.
- La latence (TTFT, TPOT, débit) sera mesurée en phase 3, sur le modèle final servi par vLLM.

**Suite** : passer `SMALL_RUN = False` pour l'entraînement complet, puis DPO (section 7).

## 7. Alignement DPO

Le DPO part du **modèle SFT** et lui apprend à préférer la réponse `chosen` à la réponse `rejected` de chaque paire.

- **Modèle de départ** : modèle de base + adaptateur SFT **fusionné** (les poids LoRA sont ajoutés aux poids du modèle). TRL calcule les log-probabilités directement depuis `lm_head`, ce qui impose de fusionner l'adaptateur SFT avant d'en ajouter un nouveau.
- **Nouvel adaptateur LoRA** entraîné par DPO. La **référence** du DPO est le modèle SFT lui-même : TRL l'obtient en désactivant l'adaptateur, sans charger un second modèle en mémoire.
- **Données** : paires françaises (MediQAl `mcqm`, vérifiées par le corrigé) et anglaises (UltraMedical, filtrées sur la qualité), mélangées pour l'entraînement, évaluées séparément par langue.
- **Reprise après redémarrage** : exécuter d'abord la section 1 (configuration), puis cette section. Le tokenizer et l'adaptateur SFT sont relus depuis `SFT_ADAPTER_DIR`. Avant la validation (7.4), exécuter aussi la cellule des fonctions de génération (section 6).

In [ ]:
# Paires DPO de la phase 1, par langue
DPO_DATASETS = {"fr": "dpo_mediqal_mcqm", "en": "dpo_ultramedical"}
DPO_COLUMNS = ["source", "lang", "pair_check", "prompt", "chosen", "rejected"]  # colonnes communes aux deux jeux

# Hyperparamètres DPO
DPO_BETA = 0.1              # écart toléré avec la référence : plus beta est grand, plus le modèle reste proche du SFT
DPO_LEARNING_RATE = 5e-5    # LoRA : ~1 % des poids entraînés ; à 5e-6 (valeur usuelle en DPO complet), les marges restaient quasi nulles
DPO_MAX_LENGTH = 1024       # prompt + réponse, en tokens
DPO_N_EVAL = 20 if SMALL_RUN else 200  # exemples du test français pour la génération (7.4)
DPO_BATCH_SIZE = 16                    # génération par lots : à baisser en cas d'OutOfMemoryError

SFT_ADAPTER_DIR = Path("../outputs") / RUN_NAME / "final"  # adaptateur SFT de la section 5
DPO_RUN_NAME = f"dpo-lora-r16-{DATA_VERSION}-beta{DPO_BETA}-lr{DPO_LEARNING_RATE}-seed{SEED}" + ("-small" if SMALL_RUN else "")
DPO_OUTPUT_DIR = Path("../outputs") / DPO_RUN_NAME  # checkpoints, adaptateur DPO et modèle final fusionné
print(f"SFT : {SFT_ADAPTER_DIR}\nDPO : {DPO_OUTPUT_DIR}")

### 7.1 Données DPO

In [ ]:
# Tokenizer sauvegardé avec l'adaptateur SFT : même gabarit qu'à l'entraînement SFT
tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_DIR)


def pair_length(example):
    # Longueur en tokens de la plus longue des deux conversations (prompt + chosen, prompt + rejected)
    return max(
        len(tokenizer(tokenizer.apply_chat_template(example["prompt"] + answer, tokenize=False), add_special_tokens=False)["input_ids"])
        for answer in (example["chosen"], example["rejected"])
    )


dpo_by_lang = {}
for lang, stem in DPO_DATASETS.items():
    files = {split: f"../data/anonymized/{stem}_{split}.anonymized.jsonl" for split in ("train", "validation", "test")}
    ds = load_dataset("json", data_files=files).select_columns(DPO_COLUMNS)
    # Paires trop longues : TRL couperait la fin de la réponse, on les écarte
    ds = ds.filter(lambda example: pair_length(example) <= DPO_MAX_LENGTH)
    if SMALL_RUN:
        for split, size in {"train": 32, "validation": 8, "test": 8}.items():
            ds[split] = ds[split].shuffle(seed=SEED).select(range(size))
    dpo_by_lang[lang] = ds
    print(f"{lang}: " + ", ".join(f"{split} {len(d)}" for split, d in ds.items()))

# Entraînement et validation : les deux langues mélangées
dpo_train = concatenate_datasets([ds["train"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
dpo_validation = concatenate_datasets([ds["validation"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
print(f"DPO : train {len(dpo_train)}, validation {len(dpo_validation)}")

In [ ]:
# Une paire telle que la verra le modèle
example = dpo_train[0]
print(tokenizer.apply_chat_template(example["prompt"], add_generation_prompt=True, tokenize=False))
print(f"--- CHOSEN :\n{example['chosen'][0]['content']}\n--- REJECTED :\n{example['rejected'][0]['content']}")

### 7.2 Modèle de départ : modèle SFT fusionné

In [ ]:
# Libère la mémoire GPU du SFT si la section 5 vient de tourner
if "trainer" in globals():
    del trainer, model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Modèle de base + adaptateur SFT fusionné : point de départ et référence du DPO
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
print(f"{sft_model.num_parameters() / 1e9:.2f} B paramètres")

### 7.3 Entraînement DPO

- **LoRA** : même configuration que pour le SFT, sur un nouvel adaptateur.
- **Une époque** : le DPO sur-apprend vite. Au-delà, le modèle mémorise les paires sans mieux généraliser.
- **Batch effectif de 16** (2 × 8) : chaque exemple contient deux réponses, donc deux fois plus de tokens qu'en SFT.
- **Learning rate de 5e-5**, avec montée progressive sur 10 % des étapes. Plus élevé qu'en DPO complet (5e-6 à 5e-7) : avec LoRA, seule une petite partie des poids est entraînée. Au premier run, 5e-6 laissait la marge des récompenses quasi nulle sur le français et la courbe n'avait pas atteint son plateau. Si `logps/chosen` chute fortement, revenir à 2e-5.

In [ ]:
dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# Suivi MLflow : même serveur que le SFT (connexion faite en section 5), expérience dédiée au DPO
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("qwen3-1.7b-dpo-lora")
mlflow.start_run(run_name=DPO_RUN_NAME)
mlflow.log_params({
    "model": MODEL_NAME, "data_version": DATA_VERSION, "sft_adapter": str(SFT_ADAPTER_DIR), "small_run": SMALL_RUN, "device": device,
    "beta": DPO_BETA, "lora_r": dpo_lora_config.r, "lora_alpha": dpo_lora_config.lora_alpha,
    **{f"pairs_{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
})

dpo_args = DPOConfig(
    output_dir=str(DPO_OUTPUT_DIR),
    run_name=DPO_RUN_NAME,
    seed=SEED,
    # DPO
    beta=DPO_BETA,
    loss_type=["sigmoid"],  # perte DPO standard
    max_length=DPO_MAX_LENGTH,
    # Optimisation
    num_train_epochs=1,
    max_steps=10 if SMALL_RUN else -1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # batch effectif = 16 paires
    learning_rate=DPO_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,  # 10 % des étapes en montée progressive du learning rate
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=5 if SMALL_RUN else 25,
    save_steps=5 if SMALL_RUN else 25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=1 if SMALL_RUN else 5,
    report_to="mlflow",
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_validation,
    processing_class=tokenizer,
    peft_config=dpo_lora_config,
)
dpo_trainer.model.print_trainable_parameters()

In [ ]:
# Reprise après interruption : dpo_trainer.train(resume_from_checkpoint=True)
dpo_result = dpo_trainer.train()
dpo_result.metrics

In [ ]:
# Sauvegarde de l'adaptateur DPO (meilleur checkpoint) et de la configuration du run
DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_trainer.save_model(str(DPO_FINAL_DIR))
tokenizer.save_pretrained(str(DPO_FINAL_DIR))

dpo_run_config = {
    "model": MODEL_NAME,
    "sft_adapter": str(SFT_ADAPTER_DIR),  # l'adaptateur DPO s'applique au modèle SFT fusionné
    "seed": SEED,
    "small_run": SMALL_RUN,
    "beta": DPO_BETA,
    "lora": {"r": dpo_lora_config.r, "alpha": dpo_lora_config.lora_alpha, "dropout": dpo_lora_config.lora_dropout},
    "data": {f"{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
    "training_args": dpo_args.to_dict(),
}
(DPO_FINAL_DIR / "run_config.json").write_text(json.dumps(dpo_run_config, indent=2, default=str), encoding="utf-8")

### 7.4 Validation DPO

1. **Préférences** sur la validation et le test de chaque langue, et sur le test anglais séparé selon la longueur (`chosen` plus longue ou plus courte que la `rejected`) : une précision très différente entre les deux révèle une préférence apprise pour la longueur plutôt que pour le fond. La *récompense implicite* d'une réponse mesure combien le modèle DPO la juge plus probable que le modèle SFT. On en tire :
   - `reward_accuracy` : part des paires où la récompense de la `chosen` dépasse celle de la `rejected` (0,5 = hasard) ;
   - `reward_margin` : écart moyen entre les deux récompenses ;
   - `logps_chosen` / `logps_rejected` : log-probabilités moyennes des deux réponses sous le modèle DPO.
2. **Réponses générées** sur le test français : modèle SFT vs modèle DPO, avec les mesures et les réponses naïves de la section 6, et le décompte des questions où un seul des deux modèles a la bonne réponse. Le test `mcqm` a un corrigé : on compare chaque réponse à la liste des bonnes réponses.

In [ ]:
# 1. Préférences, par split et par langue
# (TRL nomme ses métriques « eval_rewards/... » quel que soit le préfixe : on les renomme ici)
eval_sets = {f"{split}_{lang}": ds[split] for lang, ds in dpo_by_lang.items() for split in ("validation", "test")}

# Test anglais selon la longueur : la chosen est-elle plus longue ou plus courte que la rejected ?
test_en = dpo_by_lang["en"]["test"]
chosen_longer = [len(c[0]["content"]) > len(r[0]["content"]) for c, r in zip(test_en["chosen"], test_en["rejected"])]
eval_sets["test_en_chosen_longer"] = test_en.select([i for i, longer in enumerate(chosen_longer) if longer])
eval_sets["test_en_chosen_shorter"] = test_en.select([i for i, longer in enumerate(chosen_longer) if not longer])

dpo_rows = {}
for name, dataset in eval_sets.items():
    if len(dataset) == 0:  # petit run : un sous-ensemble peut être vide
        continue
    m = dpo_trainer.evaluate(dataset, metric_key_prefix=name)
    dpo_rows[name] = {
        "pairs": len(dataset),
        "loss": m[f"{name}_loss"],
        "reward_accuracy": m["eval_rewards/accuracies"],
        "reward_margin": m["eval_rewards/margins"],
        "logps_chosen": m["eval_logps/chosen"],
        "logps_rejected": m["eval_logps/rejected"],
    }
dpo_summary = pd.DataFrame(dpo_rows).T
dpo_summary

In [ ]:
# 2. Réponses générées sur le test français : modèle SFT (adaptateur DPO désactivé) vs modèle DPO
assert "generate_batch" in globals(), "Exécuter d'abord la cellule des fonctions de génération (section 6)"
DPO_MAX_NEW_TOKENS = 192  # réponses mcqm : listes de 2 à 5 options
DPO_ANSWER_COLUMNS = ["sft", "dpo", *NAIVE_ANSWERS]

dpo_model = dpo_trainer.model.eval()
dpo_model.config.use_cache = True  # accélère la génération

test_fr = dpo_by_lang["fr"]["test"].select(range(min(DPO_N_EVAL, len(dpo_by_lang["fr"]["test"]))))
fr_prompts = list(test_fr["prompt"])
fr_references = [chosen[0]["content"] for chosen in test_fr["chosen"]]

with dpo_model.disable_adapter():
    fr_sft_answers = generate_batch(fr_prompts, DPO_BATCH_SIZE, DPO_MAX_NEW_TOKENS, desc="Génération SFT", model=dpo_model)
fr_dpo_answers = generate_batch(fr_prompts, DPO_BATCH_SIZE, DPO_MAX_NEW_TOKENS, desc="Génération DPO", model=dpo_model)

dpo_generations = add_scores(pd.DataFrame({
    "question": [prompt[-1]["content"][-200:] for prompt in fr_prompts],
    "reference": fr_references,
    "sft": fr_sft_answers,
    "dpo": fr_dpo_answers,
    **naive_answers([chosen[0]["content"] for chosen in dpo_by_lang["fr"]["train"]["chosen"]]),
}), DPO_ANSWER_COLUMNS)
dpo_generations.to_csv(DPO_FINAL_DIR / "test_fr_generations.csv", index=False)

# Questions où un seul des deux modèles a exactement la bonne réponse (test de McNemar : ce sont elles qui comptent)
only_dpo = int(((dpo_generations["accuracy_dpo"] == 1) & (dpo_generations["accuracy_sft"] == 0)).sum())
only_sft = int(((dpo_generations["accuracy_sft"] == 1) & (dpo_generations["accuracy_dpo"] == 0)).sum())
print(f"Bonne réponse du DPO seul : {only_dpo} questions ; du SFT seul : {only_sft} questions")

dpo_comparison = score_table(dpo_generations, DPO_ANSWER_COLUMNS)
dpo_comparison

In [ ]:
# Quelques exemples à relire
dpo_generations[["question", "reference", "sft", "dpo"]].head(5)

In [ ]:
# Métriques finales dans MLflow, puis clôture du run
mlflow.log_metrics({
    f"{name}_{metric}": value
    for name, row in dpo_summary.iterrows()
    for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_fr_{metric}_{model}": dpo_comparison.loc[metric, model]
    for metric in QUALITY_METRICS
    for model in dpo_comparison.columns
})
mlflow.log_metrics({"test_fr_only_dpo_correct": only_dpo, "test_fr_only_sft_correct": only_sft})
mlflow.end_run()

### 7.5 Modèle final fusionné

Modèle de base + SFT + DPO en un seul jeu de poids complets (environ 3,4 Go en bf16), avec le tokenizer et son gabarit : c'est le modèle à servir avec vLLM en phase 3.

In [ ]:
MERGED_DIR = DPO_OUTPUT_DIR / "merged"
merged_model = dpo_trainer.model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Modèle final : {MERGED_DIR}")

## Lecture des résultats DPO

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | perte DPO d'entraînement et de validation (0,69 = aucune préférence apprise) | baissent sous 0,69 ; si `eval_loss` remonte : surapprentissage |
| `rewards/accuracies` | part des paires où la `chosen` l'emporte sur la `rejected` | monte nettement au-dessus de 0,5 |
| `rewards/margins` | écart moyen de récompense entre `chosen` et `rejected` | augmente, sans exploser |
| `logps/chosen`, `logps/rejected` | log-probabilités moyennes des deux réponses | `rejected` baisse plus que `chosen` ; si `chosen` chute fortement, baisser le learning rate ou augmenter beta |
| `{validation,test}_{fr,en}_*` | les mêmes métriques, par split et par langue | proches entre validation et test, et entre langues |
| `test_en_chosen_{longer,shorter}_*` | test anglais, selon que la `chosen` est la plus longue ou la plus courte | précisions proches ; un grand écart = préférence pour la longueur |
| `test_fr_{accuracy,option_accuracy,f1,extra_letters}_*` | lettres des réponses générées sur le test `mcqm`, comparées au corrigé (mêmes mesures et réponses naïves qu'en section 6) | dpo ≥ sft, et au-dessus de `frequente` et `tout_cocher` |
| `test_fr_only_{dpo,sft}_correct` | questions où un seul des deux modèles a exactement la bonne réponse | only_dpo > only_sft |

- Les courbes `eval_rewards/*` mélangent toutes les évaluations (TRL ignore le préfixe) : pour la comparaison finale, utiliser `{validation,test}_{fr,en}_*`.
- **Itérer** : un run par jeu d'hyperparamètres (`DPO_BETA`, `DPO_LEARNING_RATE`), avec le même `SEED`. Le nom du run les contient.

**Suite** : contrôles de sécurité (hallucinations, recommandations dangereuses) sur le modèle final, puis déploiement avec vLLM (phase 3).